# Geomagnetic Storm Forecasting Pipeline
7-day Kp index prediction from NOAA solar wind data (GradientBoosting)

In [ ]:
import json
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

#Sets up paths
DATA_DIR = pathlib.Path("data")
print(f"Data directory: {DATA_DIR}")
#Prints all files in /data if it exists or 
print(f"Files: {sorted([f.name for f in DATA_DIR.glob('*.json')] if DATA_DIR.exists() else "Error, double check data directory")}")

## 1. Data Validation

In [ ]:
#Loads raw data from JSON files
def load_json_data(filename: str) -> list | dict:
    file_path = DATA_DIR / filename
    if not file_path.exists():
        print(f"Missing: {filename}")
        return None
    with open(file_path) as f:
        return json.load(f)

# Load all available data
plasma_raw = load_json_data("plasma.json")
mag_raw = load_json_data("mag.json")
kp_raw = load_json_data("kp.json")

print("\n=== Raw Data Shapes ===")
if plasma_raw:
    print(f"Plasma: {len(plasma_raw)} rows, cols: {plasma_raw[0]}")
if mag_raw:
    print(f"Mag:    {len(mag_raw)} rows, cols: {mag_raw[0]}")
if kp_raw:
    print(f"Kp:     {len(kp_raw)} rows (dict format)")

# Convert to DataFrames
def array_to_dataframe(raw_data: list, time_col: str = "time_tag") -> pd.DataFrame:
    """Convert table-format JSON array to time-indexed DataFrame."""
    if not raw_data or len(raw_data) < 2:
        return pd.DataFrame()
    
    header = raw_data[0]
    rows = raw_data[1:]
    
    df = pd.DataFrame(rows, columns=header)
    
    # Convert time column to datetime
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors='coerce')
    
    # Convert numeric columns
    for col in df.columns:
        if col != time_col:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Set time as index and sort
    df = df.set_index(time_col).sort_index()
    return df.dropna(how='all')

plasma_df = array_to_dataframe(plasma_raw)
mag_df = array_to_dataframe(mag_raw)

# Kp has different format (dict list)
if kp_raw and isinstance(kp_raw, list) and len(kp_raw) > 0:
    if isinstance(kp_raw[0], dict):
        kp_df = pd.DataFrame(kp_raw)
        kp_df['time_tag'] = pd.to_datetime(kp_df['time_tag'], utc=True, errors='coerce')
        kp_df['Kp'] = pd.to_numeric(kp_df['Kp'], errors='coerce')
        kp_df = kp_df[['time_tag', 'Kp']].drop_duplicates('time_tag').set_index('time_tag').sort_index()
    else:
        kp_df = array_to_dataframe(kp_raw)
else:
    kp_df = pd.DataFrame()

print("\n=== Converted DataFrames ===")
print(f"Plasma: {plasma_df.shape} | dtypes:\n{plasma_df.dtypes}")
print(f"\nMag:    {mag_df.shape} | dtypes:\n{mag_df.dtypes}")
print(f"\nKp:     {kp_df.shape} | dtypes:\n{kp_df.dtypes}")

# Data quality checks
print("\n=== Data Quality ===")
print(f"Plasma nulls: {plasma_df.isnull().sum().sum()} / {plasma_df.size}")
print(f"Mag nulls:    {mag_df.isnull().sum().sum()} / {mag_df.size}")
print(f"Kp nulls:     {kp_df.isnull().sum().sum()} / {kp_df.size}")
print(f"\nTime ranges:")
print(f"  Plasma: {plasma_df.index.min()} → {plasma_df.index.max()}")
print(f"  Mag:    {mag_df.index.min()} → {mag_df.index.max()}")
print(f"  Kp:     {kp_df.index.min()} → {kp_df.index.max()}")

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=== Plasma Summary ===")
print(plasma_df.describe())

print("\n=== Magnetic Field Summary ===")
print(mag_df.describe())

print("\n=== Kp Index Summary ===")
print(kp_df.describe())

# Combine all data on time index (inner join)
merged = plasma_df.join([mag_df, kp_df], how='inner')
print(f"\n=== Merged Dataset (inner join) ===")
print(f"Shape: {merged.shape}")
print(f"Columns: {list(merged.columns)}")
print(f"Date range: {merged.index.min()} → {merged.index.max()}")
print(f"Missing values:\n{merged.isnull().sum()}")

# Plots
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle("NOAA Solar Wind & Geomagnetic Data (7-day window)", fontsize=14, fontweight='bold')

# Plasma
axes[0, 0].plot(plasma_df.index, plasma_df['speed'], linewidth=0.8)
axes[0, 0].set_title('Solar Wind Speed (km/s)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(plasma_df.index, plasma_df['density'], linewidth=0.8, color='orange')
axes[0, 1].set_title('Plasma Density (cm⁻³)')
axes[0, 1].grid(True, alpha=0.3)

# Magnetic
axes[1, 0].plot(mag_df.index, mag_df['by'], label='By', linewidth=0.8)
axes[1, 0].plot(mag_df.index, mag_df['bz'], label='Bz', linewidth=0.8)
axes[1, 0].set_title('IMF Components (nT)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(mag_df.index, mag_df['bt'], linewidth=0.8, color='red')
axes[1, 1].set_title('Total IMF Magnitude (nT)')
axes[1, 1].grid(True, alpha=0.3)

# Kp Index (target)
if not kp_df.empty:
    axes[2, 0].plot(kp_df.index, kp_df['Kp'], marker='o', markersize=3, linewidth=0.8, color='purple')
    axes[2, 0].set_title('Kp Index (Geomagnetic Activity)')
    axes[2, 0].set_ylim(0, 9)
    axes[2, 0].grid(True, alpha=0.3)

# Distribution
axes[2, 1].hist(kp_df['Kp'].dropna(), bins=20, color='purple', alpha=0.7)
axes[2, 1].set_title('Kp Index Distribution')
axes[2, 1].set_xlabel('Kp Value')
axes[2, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ EDA complete")